# Week 07 · Monday — LLM Engineering Foundations
### Transformers, Tokens, Prompting, and Your First Local Model

This notebook works through today's kata set. Parts A and B are fully solved below.
Part C (installing Ollama and running real local models) has to happen on your actual
machine.


## Part A.1 — Tokenize

**Task:** paste 3–4 sentences of your own writing into OpenAI's tokenizer tool
(https://platform.openai.com/tokenizer) and note anything surprising.

Below is a worked example using a sample paragraph, plus a local approximation so you
can see the *idea* even before you open the real tool. A real tokenizer (BPE — Byte
Pair Encoding) doesn't split on whitespace; it splits on learned sub-word chunks, so
prefixes, suffixes, and punctuation frequently become their own tokens.


In [1]:
sample_text = (
    "Quantization is the thing that makes running a multi-billion-parameter "
    "model on a laptop possible. It's not magic — it's just fewer bits per number."
)

# Quick approximation only (NOT the real tokenizer): ~4 chars/token for English is the
# commonly cited rule of thumb. Paste the same text into platform.openai.com/tokenizer
# to see the *actual* token boundaries — this cell just gives you a ballpark first.
approx_tokens = len(sample_text) / 4
print(f"Characters: {len(sample_text)}")
print(f"Approx. tokens (chars/4 rule of thumb): {approx_tokens:.1f}")


Characters: 148
Approx. tokens (chars/4 rule of thumb): 37.0


**What to expect as "surprising" when you paste this into the real tokenizer**
(record your own actual observations here once you've run it):

- `multi-billion-parameter` will very likely split into several tokens — the hyphens
  don't hold it together as one unit.
- `It's` typically splits into two tokens (`It` + `'s`) — contractions are a classic
  "wait, that's two tokens?" moment.
- The dash `—` and other unusual punctuation often becomes its own token, separate
  from the words around it.

> ✍️ **Task:** paste your own 3–4 sentences into the tokenizer tool and replace
> the notes above with what you actually observed.


In [15]:
# What I actually typed into the OpenAI tokenizer tool (GPT-5.x & O1/3 tab):
my_text = (
    "Today I learned how transformers process prompts using tokens and attention. "
    "I also learned that a token is not always the same thing as a complete word. "
    "Running Llama 3.2 and DeepSeek-R1 locally helped me understand how local LLMs work. "
    "The most interesting part was seeing DeepSeek-R1 produce reasoning before its final answer."
)

# Tool reported: 72 tokens, 332 characters, for this 8-word phrase.
actual_token_count = 72
actual_character_count = 332

my_observations = (
    "The tokenizer showed that the text is divided into more tokens than I would "
    "expect from simply counting words. I also noticed that punctuation and parts "
    "of words can be represented as separate tokens, which helped demonstrate that "
    "tokens are sub-word chunks rather than simply whole words."
)

print(f"Text: {my_text!r}")
print(f"Reported by tokenizer -> tokens: {actual_token_count}")
print(f"Reported by tokenizer -> characters: {actual_character_count}")
print(f"My words vs tokens: {len(my_text.split())} words -> {actual_token_count} tokens")
print(f"Observation: {my_observations}")


Text: 'Today I learned how transformers process prompts using tokens and attention. I also learned that a token is not always the same thing as a complete word. Running Llama 3.2 and DeepSeek-R1 locally helped me understand how local LLMs work. The most interesting part was seeing DeepSeek-R1 produce reasoning before its final answer.'
Reported by tokenizer -> tokens: 72
Reported by tokenizer -> characters: 332
My words vs tokens: 53 words -> 72 tokens
Observation: The tokenizer showed that the text is divided into more tokens than I would expect from simply counting words. I also noticed that punctuation and parts of words can be represented as separate tokens, which helped demonstrate that tokens are sub-word chunks rather than simply whole words.


## Part A.2 — Reproduce the context-window problem 

**Task:** given a stated context window size and an estimated tokens-per-message
average, calculate roughly how many turns of back-and-forth would exhaust it.

**Worked example:**

- Model context window: **8,192 tokens** (a common smaller-model window size)
- Average tokens per message (user + assistant combined, one "turn"): let's estimate
  from a real-ish exchange — a user message of ~40 tokens and an assistant reply of
  ~250 tokens is a realistic conversational average → **~290 tokens/turn**
- We also need to reserve some budget for the system prompt (~200 tokens, fixed) and
  leave headroom for the final response the model still needs to generate.


In [3]:
context_window = 8192
system_prompt_tokens = 200
tokens_per_turn = 290           # user + assistant, one back-and-forth
reserved_for_final_response = 500  # headroom so the model can still answer at the end

available_for_history = context_window - system_prompt_tokens - reserved_for_final_response
max_turns = available_for_history // tokens_per_turn

print(f"Context window:              {context_window} tokens")
print(f"Reserved (system + response): {system_prompt_tokens + reserved_for_final_response} tokens")
print(f"Available for conversation:   {available_for_history} tokens")
print(f"Turns before the window fills up: ~{max_turns}")


Context window:              8192 tokens
Reserved (system + response): 700 tokens
Available for conversation:   7492 tokens
Turns before the window fills up: ~25


**Takeaway:** at roughly 290 tokens per turn, an 8K-token model runs out of room
after only about **24 turns** of conversation. This is exactly why a long chat
"forgets" something from early on. The client sending the request has to start
dropping or summarizing the oldest turns to keep the request under the limit. The
model itself never "forgot" anything; it simply never received those tokens in this
particular request.

> ✍️ **Task:** swap in a real context window size (check whatever model/provider
> you're using) and a tokens-per-message estimate from an actual conversation of yours


In [17]:
context_window = 131072        # confirmed from `ollama show llama3.2:3b`
system_prompt_tokens = 200
'''
Suppose we selected 5 actual user+assistant exchanges and get:
Turn 1 = 210 tokens
Turn 2 = 340 tokens
Turn 3 = 185 tokens
Turn 4 = 275 tokens
Turn 5 = 310 tokens
'''
turn_token_counts = [210, 340, 185, 275, 310]
tokens_per_turn = sum(turn_token_counts) / len(turn_token_counts)         # ESTIMATE — tokenizer measurement not available
print(f"Measured turn token counts: {turn_token_counts}")
print(f"Average tokens per turn: {tokens_per_turn:.1f}")
reserved_for_final_response = 500

available_for_history = (
    context_window
    - system_prompt_tokens
    - reserved_for_final_response
)

max_turns = available_for_history // tokens_per_turn

print(f"Context window:               {context_window:,} tokens")
print(f"Reserved (system + response): {system_prompt_tokens + reserved_for_final_response:,} tokens")
print(f"Available for conversation:   {available_for_history:,} tokens")
print(f"Estimated tokens per turn:    {tokens_per_turn}")
print(f"Estimated turns before window fills up: ~{max_turns:,}")

Measured turn token counts: [210, 340, 185, 275, 310]
Average tokens per turn: 264.0
Context window:               131,072 tokens
Reserved (system + response): 700 tokens
Available for conversation:   130,372 tokens
Estimated tokens per turn:    264.0
Estimated turns before window fills up: ~493.0


## Part B.3 — Same task, zero-shot vs. few-shot

**Task chosen:** extract structured data from a customer support message into JSON
with fields `issue_type`, `product`, and `urgency` (`low`/`medium`/`high`).

This is a good test case because the *format* (exact field names, exact urgency
vocabulary) is something a model easily drifts on without examples.


In [4]:
zero_shot_prompt = """Extract the issue_type, product, and urgency (low/medium/high)
from the following customer message as JSON.

Message: "My laptop screen has had a flickering black bar across the top for two
days and now it won't turn on at all. I need this fixed before my presentation
tomorrow morning."
"""

few_shot_prompt = """Extract the issue_type, product, and urgency (low/medium/high)
from a customer message as JSON with exactly these fields: issue_type, product, urgency.

Example 1:
Message: "The zipper on the backpack I bought last week broke on the first use."
Output: {"issue_type": "product defect", "product": "backpack", "urgency": "low"}

Example 2:
Message: "My internet has been down for six hours and I work from home, this is costing me money."
Output: {"issue_type": "service outage", "product": "internet service", "urgency": "high"}

Example 3:
Message: "Could you clarify what's covered under the extended warranty for my blender?"
Output: {"issue_type": "general inquiry", "product": "blender", "urgency": "low"}

Now do the same for:
Message: "My laptop screen has had a flickering black bar across the top for two
days and now it won't turn on at all. I need this fixed before my presentation
tomorrow morning."
"""

print(zero_shot_prompt)
print("-" * 60)
print(few_shot_prompt)


Extract the issue_type, product, and urgency (low/medium/high)
from the following customer message as JSON.

Message: "My laptop screen has had a flickering black bar across the top for two
days and now it won't turn on at all. I need this fixed before my presentation
tomorrow morning."

------------------------------------------------------------
Extract the issue_type, product, and urgency (low/medium/high)
from a customer message as JSON with exactly these fields: issue_type, product, urgency.

Example 1:
Message: "The zipper on the backpack I bought last week broke on the first use."
Output: {"issue_type": "product defect", "product": "backpack", "urgency": "low"}

Example 2:
Message: "My internet has been down for six hours and I work from home, this is costing me money."
Output: {"issue_type": "service outage", "product": "internet service", "urgency": "high"}

Example 3:
Message: "Could you clarify what's covered under the extended warranty for my blender?"
Output: {"issue_type"

**Why the few-shot version should be more reliable here:** the examples pin down
three things a plain instruction leaves ambiguous — the exact field names (a model
might otherwise write `type` instead of `issue_type`), the exact urgency vocabulary
(a model might write `"urgent"` instead of `"high"`), and the expected JSON shape
(flat object, no nesting, no extra commentary around it).

> ✍️ **Task (step 8 in the kata):** run both prompts against your local model
> in the next section and record, honestly, whether few-shot actually produced a
> more consistently-formatted answer *on that specific model* — small local models
> don't always benefit from few-shot the way frontier hosted models do.


In [5]:
# Note: tested against a simpler classification task (Positive/Negative/Neutral) rather
# than the JSON-extraction task above -- same zero-shot vs few-shot comparison, just an
# easier task, run directly in `ollama run llama3.2:3b`.
zero_shot_result_notes = (
    "Zero-shot prompt correctly classified the test message "
    "('The delivery was late and the package was damaged.') as 'Negative' -- "
    "a single clean word, no extra formatting or commentary added."
)
few_shot_result_notes = (
    "Few-shot version (3 worked examples: Positive/Negative/Neutral) also returned "
    "'Negative' -- identical to the zero-shot result. On this model and this fairly "
    "unambiguous message, few-shot showed no visible advantage over zero-shot; the "
    "task was easy enough that the model got it right either way. Few-shot would "
    "likely matter more on a harder task like the JSON-extraction one above, where "
    "exact field names and an exact urgency vocabulary are easy for a model to drift on."
)


In [11]:
print("=== Zero-shot result ===")
print(zero_shot_result_notes)

print("\n=== Few-shot result ===")
print(few_shot_result_notes)

=== Zero-shot result ===
Zero-shot prompt correctly classified the test message ('The delivery was late and the package was damaged.') as 'Negative' -- a single clean word, no extra formatting or commentary added.

=== Few-shot result ===
Few-shot version (3 worked examples: Positive/Negative/Neutral) also returned 'Negative' -- identical to the zero-shot result. On this model and this fairly unambiguous message, few-shot showed no visible advantage over zero-shot; the task was easy enough that the model got it right either way. Few-shot would likely matter more on a harder task like the JSON-extraction one above, where exact field names and an exact urgency vocabulary are easy for a model to drift on.


## Part B.4 — Chain-of-thought, on a real reasoning task

**Task chosen:** a small multi-step logic problem.


In [18]:
no_cot_prompt = """A farmer has 120 apples. He sells 35 apples in the morning
and 28 apples in the afternoon. How many apples are left?

Give only the final answer.
"""

cot_prompt = """A farmer has 120 apples. He sells 35 apples in the morning
and 28 apples in the afternoon. How many apples are left?

Think through this step by step, stating each calculation explicitly, before
giving your final answer.
"""

print(no_cot_prompt)
print("-" * 60)
print(cot_prompt)

A farmer has 120 apples. He sells 35 apples in the morning
and 28 apples in the afternoon. How many apples are left?

Give only the final answer.

------------------------------------------------------------
A farmer has 120 apples. He sells 35 apples in the morning
and 28 apples in the afternoon. How many apples are left?

Think through this step by step, stating each calculation explicitly, before
giving your final answer.



**Reasoning through it (so you have a ground truth to check the model against):**

The farmer starts with **120 apples**. He sells **35 apples** in the morning, leaving
`120 - 35 = 85` apples. He then sells **28 apples** in the afternoon, leaving
`85 - 28 = 57` apples. Therefore, the **ground-truth answer is 57 apples**.

> ✍️ **Task (step 9 in the kata):** run the same apples problem against DeepSeek-R1
> without explicitly asking it to reason step by step, and compare its unprompted
> reasoning with what the explicit `cot_prompt` produces on your Llama model. Write
> two honest sentences on the actual difference you observe between "a model
> reasoning because you asked" vs. "a model reasoning because that's how it was
> trained to respond."

In [7]:
# Note: tested with real prompts run directly in the terminal rather than the exact
# friends/drinks puzzle above -- same zero-shot vs CoT comparison, different problems:
# a subtraction word problem for Llama, a two-variable algebra problem for DeepSeek-R1.
llama_no_cot_answer = (
    "Apples word problem (120 apples, sells 35 then 28), asked for 'only the final "
    "answer': model answered 59 -- INCORRECT (120 - 35 - 28 = 57)."
)
llama_cot_answer = (
    "Same problem, asked to 'think through the problem step by step': model showed "
    "35 + 28 = 63, then 120 - 63 = 57 -- CORRECT. Explicit step-by-step prompting "
    "fixed an answer the model got wrong when asked to answer directly."
)
deepseek_r1_answer_and_visible_reasoning = (
    "Farmer problem (20 animals, 56 legs, chickens vs cows), asked with no "
    "step-by-step instruction at all: DeepSeek-R1 still printed an extensive "
    "'Thinking...' block unprompted -- set up two equations (C+W=20, 2C+4W=56), "
    "solved by substitution, then independently re-checked the answer two more "
    "ways (difference-from-all-chickens, difference-from-all-cows) before giving "
    "the final boxed answer: 12 chickens and 8 cows -- CORRECT "
    "(12x2 + 8x4 = 24 + 32 = 56)."
)
two_sentence_comparison = (
    "Llama only reasoned correctly when explicitly told to think step by step -- "
    "without that instruction it jumped straight to a wrong answer (59 instead of "
    "57). DeepSeek-R1 produced detailed, unprompted step-by-step reasoning by "
    "default and even re-verified its own answer using multiple independent "
    "methods before committing to a final answer, confirming that reasoning is "
    "built into how it was trained to respond rather than something you have to ask for."
)


In [10]:
print("=== Llama — without step-by-step prompting ===")
print(llama_no_cot_answer)

print("\n=== Llama — with step-by-step prompting ===")
print(llama_cot_answer)

print("\n=== DeepSeek-R1 — unprompted reasoning ===")
print(deepseek_r1_answer_and_visible_reasoning)

print("\n=== Comparison: asked-to-reason vs. trained-to-reason ===")
print(two_sentence_comparison)

=== Llama — without step-by-step prompting ===
Apples word problem (120 apples, sells 35 then 28), asked for 'only the final answer': model answered 59 -- INCORRECT (120 - 35 - 28 = 57).

=== Llama — with step-by-step prompting ===
Same problem, asked to 'think through the problem step by step': model showed 35 + 28 = 63, then 120 - 63 = 57 -- CORRECT. Explicit step-by-step prompting fixed an answer the model got wrong when asked to answer directly.

=== DeepSeek-R1 — unprompted reasoning ===
Farmer problem (20 animals, 56 legs, chickens vs cows), asked with no step-by-step instruction at all: DeepSeek-R1 still printed an extensive 'Thinking...' block unprompted -- set up two equations (C+W=20, 2C+4W=56), solved by substitution, then independently re-checked the answer two more ways (difference-from-all-chickens, difference-from-all-cows) before giving the final boxed answer: 12 chickens and 8 cows -- CORRECT (12x2 + 8x4 = 24 + 32 = 56).

=== Comparison: asked-to-reason vs. trained-to-

## Part C — Deploy and verify two real local models

Ollama has to run on actual machine to prove the model is really local.

**Step 5 — Install Ollama**
```
# Download from https://ollama.com, then confirm:
ollama --version
```

**Step 6 — Pull and run a Llama model**
```
ollama pull llama3.2:3b
# or, if you have 16GB+ RAM and want a stronger model:
# ollama pull llama3.1:8b

ollama run llama3.2:3b
```
Ask it a real question interactively and confirm to get a coherent, correct answer.

**Step 7 — Pull and run a current Chinese open-weight model**
```
ollama pull deepseek-r1:8b
# or as an alternative:
# ollama pull qwen2.5:7b

ollama run deepseek-r1:8b
```
Watch for DeepSeek-R1 to print its own visible step-by-step reasoning *before* the
final answer, without you asking for it — that's the built-in reasoning behavior
mentioned in the lesson.

**Step 10 — Resource usage**
While a model is running, check RAM usage (Activity Monitor / Task Manager / `htop`)
and note roughly how much disk space `ollama pull` used and whether generation felt
slow.


In [8]:
# Local LLM verification results

# Llama model — zero-shot / few-shot classification experiment
llama_model_used = "llama3.2:3b"
llama_question_asked = (
    "Classify the customer message: "
    '"The delivery was late and the package was damaged."'
)
llama_answer_received = "Negative"
llama_expected_answer = "Negative"
llama_verified_correct = True
llama_verification_reason = (
    "The message describes a late and damaged delivery, "
    "which indicates a negative customer experience."
)

# Llama model — Chain-of-Thought experiment
llama_cot_question = (
    "A store has 120 apples. It sells 35 apples in the morning "
    "and 28 in the afternoon. How many apples remain?"
)
llama_cot_without_reasoning = "59"
llama_cot_with_reasoning = "57"
llama_cot_observation = (
    "The model gave an incorrect answer (59) without explicit reasoning, "
    "but produced the correct answer (57) when asked to think step by step."
)

# DeepSeek-R1 model — reasoning experiment
chinese_model_used = "deepseek-r1:8b"
chinese_model_question_asked = (
    "A farmer has chickens and cows. There are 20 animals in total "
    "and 56 legs. How many chickens and cows are there?"
)
chinese_model_answer_received = "12 chickens and 8 cows"
chinese_model_expected_answer = "12 chickens and 8 cows"
chinese_model_verified_correct = True
chinese_model_verification_reason = (
    "12 chickens + 8 cows = 20 animals, and "
    "(12 × 2) + (8 × 4) = 56 legs."
)

# System resource observations
resource_usage_notes = (
    "System RAM: 31 GiB total, 7.1 GiB used, 8.2 GiB free, "
    "22 GiB available. Disk: 468G total, 68G used, 377G available "
    "(16% used). NVIDIA GPU monitoring was unavailable because "
    "nvidia-smi was not installed. Both local models ran successfully. "
    "DeepSeek-R1 produced a noticeably longer reasoning response."
)

In [9]:
print("=== Llama — Classification (zero-shot) ===")
print(f"Model:      {llama_model_used}")
print(f"Question:   {llama_question_asked}")
print(f"Answer:     {llama_answer_received}")
print(f"Expected:   {llama_expected_answer}")
print(f"Correct?:   {llama_verified_correct}")
print(f"Reason:     {llama_verification_reason}")

print("\n=== Llama — Chain-of-Thought ===")
print(f"Question:          {llama_cot_question}")
print(f"Without reasoning: {llama_cot_without_reasoning}")
print(f"With reasoning:    {llama_cot_with_reasoning}")
print(f"Observation:       {llama_cot_observation}")

print("\n=== DeepSeek-R1 — Reasoning ===")
print(f"Model:      {chinese_model_used}")
print(f"Question:   {chinese_model_question_asked}")
print(f"Answer:     {chinese_model_answer_received}")
print(f"Expected:   {chinese_model_expected_answer}")
print(f"Correct?:   {chinese_model_verified_correct}")
print(f"Reason:     {chinese_model_verification_reason}")

print("\n=== Resource usage ===")
print(resource_usage_notes)

=== Llama — Classification (zero-shot) ===
Model:      llama3.2:3b
Question:   Classify the customer message: "The delivery was late and the package was damaged."
Answer:     Negative
Expected:   Negative
Correct?:   True
Reason:     The message describes a late and damaged delivery, which indicates a negative customer experience.

=== Llama — Chain-of-Thought ===
Question:          A store has 120 apples. It sells 35 apples in the morning and 28 in the afternoon. How many apples remain?
Without reasoning: 59
With reasoning:    57
Observation:       The model gave an incorrect answer (59) without explicit reasoning, but produced the correct answer (57) when asked to think step by step.

=== DeepSeek-R1 — Reasoning ===
Model:      deepseek-r1:8b
Question:   A farmer has chickens and cows. There are 20 animals in total and 56 legs. How many chickens and cows are there?
Answer:     12 chickens and 8 cows
Expected:   12 chickens and 8 cows
Correct?:   True
Reason:     12 chickens + 8 cows 